# 3c — Boosting moderne avec gestion native (CRISP-DM Phase 4)

Cette famille regroupe les bibliothèques de gradient boosting **modernes** qui implémentent leur propre gestion des valeurs manquantes et des variables catégorielles. C'est la **réponse « plus ingénieuse »** à la directive du prof (« si on fait du OneHotEncoder, le résultat sera mauvais ») :

- pas d'imputation en amont → le modèle apprend si NaN est un signal en soi (NaN-aware splits) ;
- pas de OneHotEncoding → pas d'explosion de dimension (79 → 256 dans `preprocessor_encoded` devient 79 → 79 ici) ;
- splits catégoriels intelligents (XGBoost ≥ 1.6 avec `enable_categorical=True`, LightGBM nativement, CatBoost via *target statistics*).

| Modèle      | Spécificité                                                                                              |
|-------------|----------------------------------------------------------------------------------------------------------|
| XGBoost     | Tree boosting standard, `enable_categorical=True` + `tree_method='hist'` pour activer la gestion native  |
| LightGBM    | Croissance **leaf-wise** (plus rapide, parfois plus profond) — *à compléter, librairie non installée*    |
| CatBoost    | **Ordered boosting** + arbres symétriques — *à compléter, librairie non installée*                       |

**Bonus pédagogique** : on entraîne deux fois XGBoost — avec `preprocessor_native` et avec `preprocessor_encoded` (OneHot) — pour quantifier le gain de la gestion native (réponse explicite à la consigne du prof).

In [ ]:
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb

In [ ]:
from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

FAMILY = 'native_boosting'

## 4.1 XGBoost — gestion native

### Hypothèses du modèle
- Boosting de gradient sur arbres : chaque arbre apprend les pseudo-résidus du modèle courant (idem GradientBoosting sklearn).
- **Différence clé** : XGBoost gère les valeurs manquantes en apprenant la **direction de défaut** (NaN-aware splits) — la donnée manquante devient un signal apprenable.
- Avec `enable_categorical=True` + `tree_method='hist'`, XGBoost fait aussi des splits sur les `pd.Categorical` directement (algorithme spécifique pour partitionner les niveaux catégoriels).

In [ ]:
t0 = time.time()
xgb_native_pipe = Pipeline([
    ('preprocessor', preprocessor_native),
    ('model', XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        enable_categorical=True, tree_method='hist',
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    )),
])
xgb_native_pipe.fit(X_train, y_train_log)
xgb_native_fit_s = time.time() - t0

xgb_native_pred = xgb_native_pipe.predict(X_test)
xgb_native_holdout = rmsle_score(y_test_log, xgb_native_pred)
xgb_native_cv = cv_rmsle(xgb_native_pipe, X_train, y_train_log)

print(f"XGBoost native — CV RMSLE: {xgb_native_cv:.4f}   Holdout RMSLE: {xgb_native_holdout:.4f}   fit_time: {xgb_native_fit_s:.2f}s")
predicted_vs_actual_plot(y_test_log, xgb_native_pred, title=f"XGBoost native — RMSLE: {xgb_native_holdout:.4f}")
plt.show()

publish_result(FAMILY, 'XGBoost_native', cv_rmsle=xgb_native_cv, holdout_rmsle=xgb_native_holdout, fit_time_s=xgb_native_fit_s,
               params={'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 4, 'enable_categorical': True})

## 4.2 XGBoost — variante OneHot (pour comparaison)

On entraîne le **même** XGBoost (mêmes hyperparamètres) sur le préprocesseur OneHot. La différence de RMSLE et de temps d'entraînement isolent l'apport pédagogique de la gestion native — la réponse concrète à la directive du prof.

In [ ]:
t0 = time.time()
xgb_enc_pipe = Pipeline([
    ('preprocessor', preprocessor_encoded),
    ('model', XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    )),
])
xgb_enc_pipe.fit(X_train, y_train_log)
xgb_enc_fit_s = time.time() - t0

xgb_enc_pred = xgb_enc_pipe.predict(X_test)
xgb_enc_holdout = rmsle_score(y_test_log, xgb_enc_pred)
xgb_enc_cv = cv_rmsle(xgb_enc_pipe, X_train, y_train_log)

print(f"XGBoost OneHot — CV RMSLE: {xgb_enc_cv:.4f}   Holdout RMSLE: {xgb_enc_holdout:.4f}   fit_time: {xgb_enc_fit_s:.2f}s")
print()
print(f"Comparaison native vs OneHot (mêmes hyperparams) :")
print(f"  CV RMSLE      : native {xgb_native_cv:.4f}   |  OneHot {xgb_enc_cv:.4f}   |  delta {xgb_enc_cv - xgb_native_cv:+.4f}")
print(f"  Holdout RMSLE : native {xgb_native_holdout:.4f}   |  OneHot {xgb_enc_holdout:.4f}   |  delta {xgb_enc_holdout - xgb_native_holdout:+.4f}")
print(f"  fit_time      : native {xgb_native_fit_s:.2f}s   |  OneHot {xgb_enc_fit_s:.2f}s")

publish_result(FAMILY, 'XGBoost_onehot', cv_rmsle=xgb_enc_cv, holdout_rmsle=xgb_enc_holdout, fit_time_s=xgb_enc_fit_s,
               params={'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 4, 'enable_categorical': False},
               notes='Variante pédagogique pour comparaison avec XGBoost_native')

## 4.3 XGBoost — optimisation Optuna

On optimise les hyperparamètres du XGBoost natif avec Optuna sur 30 trials (TPE sampler, 5-fold CV interne). Les 4 graphiques du cours (`optimization_history`, `param_importances`, `contour`, `slice`) sont produits ci-dessous.

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    pipe = Pipeline([
        ('preprocessor', preprocessor_native),
        ('model', XGBRegressor(**params, enable_categorical=True, tree_method='hist',
                               random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)),
    ])
    rmsle = -cross_val_score(pipe, X_train, y_train_log, cv=5,
                             scoring='neg_root_mean_squared_error', n_jobs=-1).mean()
    return rmsle

t0 = time.time()
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(xgb_objective, n_trials=30, show_progress_bar=False)
study_fit_s = time.time() - t0

print(f"Optuna terminé en {study_fit_s:.1f}s — meilleur CV RMSLE: {study.best_value:.4f}")
print(f"Meilleurs hyperparamètres :")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

### Visualisations Optuna (exigence du cours 09_Hyperparameter_Tuning)

Quatre graphiques standard : historique, importance des hyperparamètres, contour 2D et slice.

In [ ]:
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title("Optuna — historique des essais")
plt.show()

optuna.visualization.matplotlib.plot_param_importances(study)
plt.title("Optuna — importance des hyperparamètres")
plt.show()

try:
    optuna.visualization.matplotlib.plot_contour(study, params=['learning_rate', 'max_depth'])
    plt.title("Optuna — contour learning_rate × max_depth")
    plt.show()
except Exception as e:
    print(f"plot_contour skipped: {e}")

try:
    optuna.visualization.matplotlib.plot_slice(study, params=['learning_rate', 'max_depth', 'n_estimators'])
    plt.show()
except Exception as e:
    print(f"plot_slice skipped: {e}")

In [ ]:
t0 = time.time()
xgb_tuned_pipe = Pipeline([
    ('preprocessor', preprocessor_native),
    ('model', XGBRegressor(**study.best_params, enable_categorical=True, tree_method='hist',
                           random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)),
])
xgb_tuned_pipe.fit(X_train, y_train_log)
xgb_tuned_fit_s = time.time() - t0

xgb_tuned_pred = xgb_tuned_pipe.predict(X_test)
xgb_tuned_holdout = rmsle_score(y_test_log, xgb_tuned_pred)
xgb_tuned_cv = float(study.best_value)

print(f"XGBoost (tuned) — CV RMSLE: {xgb_tuned_cv:.4f}   Holdout RMSLE: {xgb_tuned_holdout:.4f}   fit_time (final): {xgb_tuned_fit_s:.2f}s   optuna: {study_fit_s:.1f}s")
predicted_vs_actual_plot(y_test_log, xgb_tuned_pred, title=f"XGBoost (Optuna tuned) — RMSLE: {xgb_tuned_holdout:.4f}")
plt.show()

publish_result(FAMILY, 'XGBoost_tuned', cv_rmsle=xgb_tuned_cv, holdout_rmsle=xgb_tuned_holdout,
               fit_time_s=xgb_tuned_fit_s, params=study.best_params,
               notes=f'Optuna 30 trials over {study_fit_s:.0f}s')

## 4.4 LightGBM — TODO

**Librairie non installée** dans l'environnement actuel (cf. `requirements.txt`). À ajouter :

```bash
pip install lightgbm
```

Patron de code à utiliser une fois disponible :

```python
from lightgbm import LGBMRegressor
lgb_pipe = Pipeline([
    ('preprocessor', preprocessor_native),
    ('model', LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        categorical_feature='auto',  # détecte les pd.Categorical automatiquement
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])
```

LightGBM se distingue de XGBoost par sa croissance **leaf-wise** (au lieu de level-wise), ce qui le rend plus rapide mais potentiellement plus enclin à surapprendre sur petits jeux.

## 4.5 CatBoost — TODO

**Librairie non installée** dans l'environnement actuel. À ajouter :

```bash
pip install catboost
```

Patron de code à utiliser une fois disponible :

```python
from catboost import CatBoostRegressor
cat_cols = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
cb_pipe = Pipeline([
    ('preprocessor', preprocessor_native),
    ('model', CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=6,
        cat_features=cat_cols,  # CatBoost veut la liste explicite
        random_seed=RANDOM_STATE, verbose=False,
    )),
])
```

CatBoost utilise l'**ordered boosting** (évite le biais de cible classique du target encoding) et des **arbres symétriques** (toutes les feuilles à la même profondeur), ce qui le rend particulièrement robuste sur petits jeux.

## 4.6 Comparaison intra-famille

Comparaison entre les variantes de XGBoost. Une fois LightGBM/CatBoost installés et entraînés, le graphique se remplira automatiquement.

In [ ]:
family_path = RESULTS_DIR / f'family_{FAMILY}.json'
fam = pd.DataFrame(json.loads(family_path.read_text()))
fam = fam.sort_values('holdout_rmsle').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(fam['model'], fam['holdout_rmsle'], color=sns.color_palette('viridis', len(fam)))
for bar, v in zip(bars, fam['holdout_rmsle']):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.002, f"{v:.4f}", ha='center', fontweight='bold')
ax.set_ylabel('RMSLE (holdout)')
ax.set_title("Famille « native_boosting » — RMSLE par modèle")
ax.set_ylim(0, fam['holdout_rmsle'].max() * 1.15)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

display(fam[['model', 'cv_rmsle', 'holdout_rmsle', 'fit_time_s', 'notes']])